In [ ]:
# === Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Install necessary packages (only run once) ===
!pip install -q mne plotly colorama pyphen

# === Imports ===
import os
import gc
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import pyphen
import mne
import plotly.graph_objects as go
from IPython.display import display
from colorama import Fore, Style, init

# === Colorama init (for colored terminal printouts) ===
init(autoreset=True)


# Hypothesis:
Null Hypothesis (H₀):
ERP and acoustic features (e.g., speech rate, inter-word interval, MFCCs, F0, etc.) do not predict participants’ perception accuracy better than chance.


Alternative Hypothesis (H₁):
ERP and acoustic features do predict perception accuracy significantly better than chance.



# Table of Contents
- Hypotheses
- Data Preprocessing
- Feature Engineering
- Modeling & Classification
- Results & Interpretation



# 1. Data Preprocessing

## Behavioral Data Preprocessing




In [ ]:
import os
import pandas as pd

# === 1. Parse Sentences_List1.txt ===
def parse_sentences_from_txt(txt_file):
    with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    data = []
    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                idx, rest = line.split('.', 1)
                words = rest.strip().split()

                word1_raw = None
                function_word = None
                word2 = None

                for i, word in enumerate(words):
                    if '|' in word:
                        word1_raw = word
                        if i + 1 < len(words):
                            function_word = words[i + 1]
                        if i + 2 < len(words):
                            word2 = words[i + 2].replace('|', '')
                        break

                if word1_raw and function_word:
                    word1_clean = word1_raw.replace('|', '').lower()
                    function_word = function_word.lower()
                    full_sentence = ' '.join(w.replace('|', '') for w in words)

                    data.append({
                        'Index': int(idx),
                        'Word1': word1_clean,
                        'FunctionWord': function_word,
                        'Word2': word2.lower() if word2 else None,
                        'Full_Sentence': full_sentence.lower()
                    })

            except Exception as e:
                print(f"⚠️ Failed to parse line: {line}\n{e}")
                continue

    return pd.DataFrame(data)

# === 2. Check response logic ===
def check_sequence_after_word1(row, sentence_df):
    stim_text = str(row.get("StimTest.RESP", "")).lower().strip()
    stim_words = stim_text.split()

    idx = row.get("Index")
    match = sentence_df[sentence_df["Index"] == idx]
    if match.empty:
        return None

    word1 = match.iloc[0]["Word1"]
    func_word = match.iloc[0]["FunctionWord"]
    word2 = match.iloc[0]["Word2"]

    try:
        i = stim_words.index(word1)
    except ValueError:
        return None  # word1 not found

    if i + 1 < len(stim_words) and stim_words[i + 1] == func_word:
        return 0
    if i + 1 < len(stim_words) and stim_words[i + 1] == word2:
        return 1
    return None

# === 3. Smart CSV reader ===
def smart_read_csv(filepath):
    for enc in ['utf-8', 'utf-16', 'windows-1252']:
        for delim in [',', '\t', ';']:
            try:
                df = pd.read_csv(filepath, delimiter=delim, encoding=enc, engine='python', on_bad_lines='skip')
                if df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    return None

# === 4. Main function with save ===
def print_evaluation_of_participants(root_dir, sentence_file):
    sentence_df = parse_sentences_from_txt(sentence_file)
    all_dfs = []  # Collect for saving

    for participant_num in range(2, 28):
        folder_path = os.path.join(root_dir, str(participant_num))
        if not os.path.isdir(folder_path):
            print(f"❌ Folder not found: {folder_path}")
            continue

        print(f"\n👤 Processing Participant {participant_num}")

        for fname in os.listdir(folder_path):
            if fname.lower().startswith("speechrate") and fname.lower().endswith(".csv"):
                input_path = os.path.join(folder_path, fname)
                df = smart_read_csv(input_path)

                if df is None:
                    print(f"❌ Could not read: {fname}")
                    continue

                df.rename(columns={"WavFiler": "Index", "CellT": "Condition"}, inplace=True)

                if not all(c in df.columns for c in ["Index", "Condition", "StimTest.RESP"]):
                    print(f"⚠️ Unexpected columns in {fname}")
                    continue

                df["Index"] = df["Index"].astype("Int64", errors="ignore")
                df = pd.merge(df, sentence_df, on="Index", how="left")

                df["Correct_Sequence (0/1)"] = df.apply(lambda row: check_sequence_after_word1(row, sentence_df), axis=1)
                df["Correct_Sequence (0/1)"] = df["Correct_Sequence (0/1)"].astype("Int64")
                df["Include_For_Stats"] = df["Correct_Sequence (0/1)"].isin([0, 1])
                df["Participant"] = participant_num

                all_dfs.append(df)
                print(df[["Index", "Condition", "StimTest.RESP", "Correct_Sequence (0/1)", "Include_For_Stats"]].head(10))
                break

    # Combine and save
    if all_dfs:
        full_df = pd.concat(all_dfs, ignore_index=True)
        save_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_includeNA.csv"
        full_df.to_csv(save_path, index=False)
        print(f"\n✅ Saved full dataset with NAs: {save_path}")
    else:
        print("⚠️ No valid participant data to save.")

# === 5. Run ===
if __name__ == "__main__":
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    sentence_file = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"
    print_evaluation_of_participants(root_dir, sentence_file)






| Value  | Meaning                                                         |
| ------ | --------------------------------------------------------------- |
| `0`    | **Correct** — The function word followed the expected `Word1` |
| `1`    | **Incorrect** — The function word **did not** follow `Word1`  |
| `<NA>` | **Not applicable** — One of the following:                   |
|        | - The response was too short                                    |
                    |
|        | - The sentence was malformed or incomplete                      |


## Load Onset Times



In [ ]:
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"
df = pd.read_csv(csv_path)
print(df.head())



### Check If the Onset Times are in ms or seconds

In [ ]:


# === Set paths ===
audio_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli"

# === Check a sample audio file ===
sample_file = "SR129.wav"
audio_path = os.path.join(audio_dir, sample_file)

# === Load audio and print duration ===
y, sr = librosa.load(audio_path, sr=16000)  # 16kHz
duration = len(y) / sr
print(f"\n🔊 '{sample_file}' duration: {duration:.2f} seconds")

# === Check if onset values are in ms or sec ===
onset_sample_ms = df.loc[0, "critical"]  # or any other column
onset_sample_sec = onset_sample_ms / 1000
print(f"Sample 'critical' onset (converted): {onset_sample_sec:.3f} sec")

# === Conclusion ===
if onset_sample_sec < duration:
    print("✅ Onset values are in milliseconds.")
else:
    print("⚠️ Onset values might already be in seconds — please double check.")



## Merge Behavioral & Timing Data

In [ ]:
import pandas as pd

# === Load behavioral data ===
behavior_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_includeNA.csv"
df_behavior = pd.read_csv(behavior_path)

# === Load onset timing data ===
onset_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"
df_onset = pd.read_csv(onset_path)

# === Clean column names ===
df_behavior.columns = df_behavior.columns.str.strip()
df_onset.columns = df_onset.columns.str.strip()

# === Rename onset columns for consistency ===
df_onset = df_onset.rename(columns={
    "sentence#": "Index",
    "Word1": "Word1_Onset",
    "critical": "FunctionWord_Onset",
    "Word2": "Word2_Onset",
    "NorE": "Condition"
})

# === Ensure 'Index' and 'Condition' are of the same type and consistent ===
df_behavior["Index"] = df_behavior["Index"].astype(int)
df_onset["Index"] = df_onset["Index"].astype(int)

df_behavior["Condition"] = df_behavior["Condition"].str.upper().str.strip()
df_onset["Condition"] = df_onset["Condition"].str.upper().str.strip()

# === Merge on Index + Condition to avoid duplication ===
df_merged = pd.merge(df_behavior, df_onset, on=["Index", "Condition"], how="left")

# ✅ Convert key columns to nullable integers
int_cols = ["Correct_Sequence (0/1)", "Word1_Onset", "FunctionWord_Onset", "Word2_Onset"]
for col in int_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].astype("Int64")

# === Save merged dataframe ===
save_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral_With_Correct_Onsets.csv"
df_merged.to_csv(save_path, index=False)

print(f"✅ Merged and saved to: {save_path}")
print("🧾 Preview:")
display(df_merged.head(100))


#2. Feature Engineering

### Feature Preprocessing

###Split the regions for each sentence

In [ ]:

# === Load and parse Sentences_List1.txt ===
def parse_sentence_components(txt_path):
    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    structured_data = []

    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                index_part, sentence = line.split('.', 1)
                index = int(index_part.strip())

                # Split the sentence into chunks using |
                chunks = sentence.strip().split('|')

                # Safety check
                if len(chunks) < 4:
                    print(f"⚠️ Skipping line {index}: not enough | segments.")
                    continue

                # Extract components
                preceding_context = chunks[0].strip()
                target_raw = chunks[1].strip()
                following_context = chunks[2].strip()
                extra_context = chunks[3].strip()

                # Target region is structured: Word1 + Function Word + start of Word2
                target_tokens = target_raw.split()
                if len(target_tokens) < 3:
                    print(f"⚠️ Skipping line {index}: target region has < 3 tokens.")
                    continue

                word1 = target_tokens[0]
                function_word = target_tokens[1]
                word2_start = target_tokens[2]

                target_region = f"{word1} {function_word} {word2_start}"

                structured_data.append({
                    "Index": index,
                    "Preceding_Context": preceding_context,
                    "Target_Region": target_region,
                    "Following_Context": following_context,
                    "Extra_Context": extra_context
                })

            except Exception as e:
                print(f"❌ Error parsing line: {line}\n{e}")
                continue

    return pd.DataFrame(structured_data)

# === Path to your sentence file ===
sentence_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"

# === Run the parser ===
df_sentences = parse_sentence_components(sentence_path)

# === Save parsed output ===
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
df_sentences.to_csv(csv_path, index=False)

# Optional: save as Pickle (faster for later use in Python)
pkl_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.pkl"
df_sentences.to_pickle(pkl_path)

# === Display the first few rows ===
from IPython.display import display
display(df_sentences.head(10))






### Feature 1: Speech Rate of Preceding Context

In [ ]:

# === File paths ===
parsed_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
onset_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"

# === Load parsed sentences ===
df_sentences = pd.read_csv(parsed_path)

# === Load onset times and rename columns for consistency ===
df_onsets = pd.read_csv(onset_path)
df_onsets.rename(columns={
    "sentence#": "Index",
    "critical": "Target_Onset_ms"
}, inplace=True)

# === Merge on Index ===
df = pd.merge(df_sentences, df_onsets, on="Index")

# === Convert onset time to seconds ===
df["Preceding_Duration_sec"] = df["Target_Onset_ms"] / 1000.0

# === Count syllables in Preceding_Context ===
dic = pyphen.Pyphen(lang='en')

def count_syllables(text):
    return sum(len(dic.inserted(word).split('-')) for word in text.split())

df["Preceding_Syllables"] = df["Preceding_Context"].apply(count_syllables)

# === Compute Distal Speech Rate ===
df["SpeechRate_Preceding"] = df["Preceding_Syllables"] / df["Preceding_Duration_sec"]

# === Display result ===
display(df[["Index", "Preceding_Context", "Preceding_Syllables", "Preceding_Duration_sec", "SpeechRate_Preceding"]].head(10))

# === Save final DataFrame to CSV ===
output_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_with_DistalSpeechRate.csv"
df.to_csv(output_path, index=False)
print(f"Data saved to: {output_path}")



In [ ]:
def load_onset_features(base_path):
    all_features = []
    sentence_ids = []

    for sentence_folder in sorted(os.listdir(base_path)):
        sentence_path = os.path.join(base_path, sentence_folder)

        if not os.path.isdir(sentence_path):
            continue

        sentence_features = []

        # List of expected files
        files = [
            "word_step_regressor.npy",
            "word_step_duration_regressor.npy",
            "word_duration_regressor.npy",
            "word_binary.npy",
            "phoneme_step_regressor.npy",
            "phoneme_step_duration_regressor.npy",
            "phoneme_duration_regressor.npy",
            "phoneme_binary.npy"
        ]

        for file_name in files:
            file_path = os.path.join(sentence_path, file_name)
            if os.path.exists(file_path):
                data = np.load(file_path)
                sentence_features.append(data.flatten())
            else:
                print(f"⚠️ Missing file: {file_path}, filling with zeros")
                sentence_features.append(np.zeros(10))  # placeholder

        # CSV files for onsets
        csv_files = ["word_onsets.csv", "phoneme_onsets.csv"]
        for csv_name in csv_files:
            csv_path = os.path.join(sentence_path, csv_name)
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                sentence_features.append(df.values.flatten())
            else:
                print(f"⚠️ Missing CSV: {csv_path}, filling with zeros")
                sentence_features.append(np.zeros(10))

        combined_features = np.concatenate(sentence_features)
        all_features.append(combined_features)
        sentence_ids.append(sentence_folder)

    # --- PAD FEATURE VECTORS ---
    max_len = max(len(f) for f in all_features)
    padded_features = np.zeros((len(all_features), max_len))

    for i, f in enumerate(all_features):
        padded_features[i, :len(f)] = f  # pad remaining with zeros

    return padded_features, sentence_ids

# Load onset features
X_onset, sentence_ids = load_onset_features(onset_dir)
print("✅ Onset features loaded:", X_onset.shape)



###  Acoustic Features Predicting Perception


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# === 1. Load behavioral data ===
df_merged = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral_With_Correct_Onsets.csv")

# Load distal speech rate data
df_rate = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_with_DistalSpeechRate.csv")
df_rate.columns = df_rate.columns.str.strip()

# Ensure consistent formatting
df_rate["Index"] = df_rate["Index"].astype(int)
df_rate["Condition"] = df_rate["NorE"].str.upper().str.strip()
df_merged["Index"] = df_merged["Index"].astype(int)
df_merged["Condition"] = df_merged["Condition"].str.upper().str.strip()

# Merge speech rate into behavioral data
df_merged = df_merged.drop(columns=[col for col in df_merged.columns if "SpeechRate_Preceding" in col], errors="ignore")
df_merged = df_merged.merge(
    df_rate[["Index", "Condition", "SpeechRate_Preceding"]],
    on=["Index", "Condition"],
    how="left"
)

# Handle missing labels
print("Missing labels:", df_merged["Correct_Sequence (0/1)"].isna().sum())
df_merged = df_merged.dropna(subset=["Correct_Sequence (0/1)"])
df_merged["Correct_Sequence (0/1)"] = df_merged["Correct_Sequence (0/1)"].astype(int)
print("✅ Behavioral and speech rate data loaded:", df_merged.shape)

# Create unique sentence ID
df_merged["Sentence_ID"] = df_merged["Index"].astype(str) + "_" + df_merged["Condition"]

# === 2. Load ERP features ===
erp_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Features.csv"
df_erp = pd.read_csv(erp_path)
df_erp.columns = df_erp.columns.str.strip()

print("✅ ERP features loaded:", df_erp.shape)

# Merge ERP into behavioral data
df_merged = df_merged.merge(df_erp, on=["Participant", "Index"], how="left")

# === 3. Load summarized onset features ===
onset_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/onset_vectors/onset_vectors"

def extract_summary_features(base_path):
    all_features = []

    files = [
        "word_step_regressor.npy",
        "word_step_duration_regressor.npy",
        "word_duration_regressor.npy",
        "word_binary.npy",
        "phoneme_step_regressor.npy",
        "phoneme_step_duration_regressor.npy",
        "phoneme_duration_regressor.npy",
        "phoneme_binary.npy"
    ]
    csv_files = ["word_onsets.csv", "phoneme_onsets.csv"]

    for sentence_folder in sorted(os.listdir(base_path)):
        sentence_path = os.path.join(base_path, sentence_folder)
        if not os.path.isdir(sentence_path):
            continue

        features = {"Sentence_ID": sentence_folder}

        for file_name in files:
            file_path = os.path.join(sentence_path, file_name)
            if os.path.exists(file_path):
                data = np.load(file_path).flatten()
                features[f"{file_name}_mean"] = np.mean(data)
                features[f"{file_name}_std"] = np.std(data)
            else:
                features[f"{file_name}_mean"] = 0
                features[f"{file_name}_std"] = 0

        for csv_name in csv_files:
            csv_path = os.path.join(sentence_path, csv_name)
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                values = df.values.flatten()
                features[f"{csv_name}_mean"] = np.mean(values)
                features[f"{csv_name}_std"] = np.std(values)
            else:
                features[f"{csv_name}_mean"] = 0
                features[f"{csv_name}_std"] = 0

        all_features.append(features)

    return pd.DataFrame(all_features)

df_onset = extract_summary_features(onset_dir)
print("✅ Onset summary features loaded:", df_onset.shape)

# Merge onset features into behavioral data
df_merged = df_merged.merge(df_onset, on="Sentence_ID", how="left")
print("✅ Full dataset merged:", df_merged.shape)

# === 4. Prepare features (select numeric only) ===
feature_cols = df_merged.drop(columns=["Participant", "Index", "Condition", "Correct_Sequence (0/1)", "Sentence_ID"]).select_dtypes(include=[np.number]).columns.tolist()

X = df_merged[feature_cols].fillna(0).values
y = df_merged["Correct_Sequence (0/1)"].values

print("✅ Final feature matrix:", X.shape)
print("✅ Target vector:", y.shape)

# === 5. Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# === 6. Scale features ===
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === 7. Train XGBoost ===
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_xgb.fit(X_train, y_train)
print("✅ XGBoost training complete!")

# === 8. Train Logistic Regression ===
model_lr = LogisticRegression(max_iter=500)
model_lr.fit(X_train, y_train)
print("✅ Logistic Regression training complete!")

# === 9. Evaluate both models ===
print("\n📊 XGBoost Classification Report:")
print(classification_report(y_test, model_xgb.predict(X_test)))

print("\n📊 Logistic Regression Classification Report:")
print(classification_report(y_test, model_lr.predict(X_test)))

# === 10. Feature Importance (XGBoost) ===
importances = model_xgb.feature_importances_
feature_importance = pd.DataFrame({"Feature": feature_cols, "Importance": importances})
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

print("\n🔥 Top XGBoost Features:")
print(feature_importance.head(10))

# Plot feature importance
sns.barplot(data=feature_importance.head(15), x="Importance", y="Feature")
plt.title("Top 15 XGBoost Feature Importances")
plt.show()

# === 11. Logistic Regression Coefficients ===
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model_lr.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print("\n🧠 Logistic Regression Coefficients:")
print(coef_df.head(10))



In [ ]:
def load_onset_features_avg(base_path):
    all_feature_avgs = []
    sentence_ids = []

    for sentence_folder in sorted(os.listdir(base_path)):
        sentence_path = os.path.join(base_path, sentence_folder)

        if not os.path.isdir(sentence_path):
            continue

        feature_avgs = []

        # List of expected files
        files = [
            "word_step_regressor.npy",
            "word_step_duration_regressor.npy",
            "word_duration_regressor.npy",
            "word_binary.npy",
            "phoneme_step_regressor.npy",
            "phoneme_step_duration_regressor.npy",
            "phoneme_duration_regressor.npy",
            "phoneme_binary.npy"
        ]

        for file_name in files:
            file_path = os.path.join(sentence_path, file_name)
            if os.path.exists(file_path):
                data = np.load(file_path)
                feature_avgs.append(np.mean(data))  # average value
            else:
                print(f"⚠️ Missing file: {file_path}, filling with zero")
                feature_avgs.append(0.0)

        # CSV files for onsets
        csv_files = ["word_onsets.csv", "phoneme_onsets.csv"]
        for csv_name in csv_files:
            csv_path = os.path.join(sentence_path, csv_name)
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                feature_avgs.append(np.mean(df.values))  # average of values
            else:
                print(f"⚠️ Missing CSV: {csv_path}, filling with zero")
                feature_avgs.append(0.0)

        all_feature_avgs.append(feature_avgs)
        sentence_ids.append(sentence_folder)

    return np.array(all_feature_avgs), sentence_ids

# Then load averages
X_onset_avg, sentence_ids = load_onset_features_avg(onset_dir)
print("✅ Onset average features loaded:", X_onset_avg.shape)


#3. Inference:

## Accoustic Features Only

In [ ]:
# Assuming you have X_onset_avg and sentence_ids from the averaged onset feature loader
# Also assuming df_merged has a 'Sentence_ID' column to join on sentence_ids

import numpy as np
import pandas as pd

# Create a DataFrame for averaged onset features
df_onset_avg = pd.DataFrame(X_onset_avg, columns=[
    "avg_word_step_regressor",
    "avg_word_step_duration_regressor",
    "avg_word_duration_regressor",
    "avg_word_binary",
    "avg_phoneme_step_regressor",
    "avg_phoneme_step_duration_regressor",
    "avg_phoneme_duration_regressor",
    "avg_phoneme_binary",
    "avg_word_onsets",
    "avg_phoneme_onsets"
])
df_onset_avg["Sentence_ID"] = sentence_ids

# Merge onset averages with behavioral data on Sentence_ID
df_model = df_merged.merge(df_onset_avg, on="Sentence_ID", how="inner")

# Prepare feature matrix X and target y
feature_cols = df_onset_avg.columns.drop("Sentence_ID").tolist()
X = df_model[feature_cols].values
y = df_model["Correct_Sequence (0/1)"].values

# Then continue with your train/test split and logistic regression code as you wrote it:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("✅ Feature matrix for Logistic Regression:", X.shape)
print("✅ Target vector:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("📊 Classification Report (Logistic Regression):\n")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\n🧠 Top Most Influential Features (Logistic Regression):")
display(coef_df)



In [ ]:
from xgboost import XGBClassifier

# Train/test split (reuse X and y from onset features)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_xgb = xgb_model.predict(X_test)

print("📊 Classification Report (XGBoost):\n")
print(classification_report(y_test, y_pred_xgb))

# Confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap="Greens",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - XGBoost")
plt.show()

# Feature importance
feature_importance = pd.DataFrame({
    "Feature": [f"feature_{i}" for i in range(X.shape[1])],
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n🔥 Top 20 Most Important Features (XGBoost):")
display(feature_importance.head(20))


Summary:

## ERP

In [ ]:
import os
import mne
import numpy as np
import pandas as pd

# === Configuration ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
output_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Features.csv'


# Time windows (in seconds)
n1_window = (0.09, 0.13)     # ~100 ms
n280_window = (0.25, 0.32)   # ~280 ms

# Optional: specify channels to average over (e.g., central)
# If None, will average over all EEG channels
selected_channels = None  # e.g., ['Cz', 'FCz', 'CPz']

# === Processing loop ===
all_erp_data = []

subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    clean_epo_path = os.path.join(root_dir, subj, f"{subj}_O1_epo_clean.fif")

    if not os.path.exists(clean_epo_path):
        print(f"❌ No cleaned epochs for subject {subj}, skipping.")
        continue

    print(f"📥 Processing subject {subj}...")

    try:
        epochs = mne.read_epochs(clean_epo_path, preload=True)
        times = epochs.times
        data = epochs.get_data()  # shape: (n_trials, n_channels, n_times)

        # Subset channels if specified
        if selected_channels:
            picks = mne.pick_channels(epochs.ch_names, selected_channels)
            data = data[:, picks, :]
        else:
            # Average over all EEG channels (not EOG or misc)
            picks = mne.pick_types(epochs.info, eeg=True)
            data = data[:, picks, :]

        # Get indices for N1 and N280 time windows
        n1_idx = np.where((times >= n1_window[0]) & (times <= n1_window[1]))[0]
        n280_idx = np.where((times >= n280_window[0]) & (times <= n280_window[1]))[0]

        # Extract mean amplitude per trial per window
        n1_amp = data[:, :, n1_idx].mean(axis=(1, 2))
        n280_amp = data[:, :, n280_idx].mean(axis=(1, 2))
        print(f"Size n1_amp: {n1_amp.shape}.\n Size n280_amp: {n280_amp.shape}.")

        # Trial-level dataframe
        n_trials = len(n1_amp)
        df = pd.DataFrame({
            'Participant': [int(subj)] * n_trials,
            'Index': np.arange(n_trials),
            'N1_amplitude': n1_amp,
            'N280_amplitude': n280_amp
        })
        all_erp_data.append(df)

    except Exception as e:
        print(f"⚠️ Error processing subject {subj}: {e}")
        continue

# === Combine and save ===
erp_df = pd.concat(all_erp_data, ignore_index=True)
erp_df.to_csv(output_path, index=False)

print(f"\n✅ Saved ERP feature file: {output_path}")
display(erp_df.head(40))


##ERP + 2 Features

 ### Merge ERP + Acoustic Data

In [ ]:
import pandas as pd
import numpy as np

# === Step 1: Load behavioral data ===
df_merged = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral_With_Correct_Onsets.csv")
df_merged.columns = df_merged.columns.str.strip()

# Keep only required columns
features_df = df_merged[[
    "Participant",
    "Index",
    "Condition",
    "Correct_Sequence (0/1)"
]].dropna()

# Convert target column to integer
features_df["Correct_Sequence (0/1)"] = features_df["Correct_Sequence (0/1)"].astype(int)
print("✅ Behavioral data loaded:", features_df.shape)
display(features_df.head())

# === Step 2: Load ERP features ===
erp_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Features.csv"
erp_df = pd.read_csv(erp_path)
erp_df.columns = erp_df.columns.str.strip()

print("✅ ERP features loaded:", erp_df.shape)
display(erp_df.head())

# === Step 3: Merge behavioral + ERP ===
combined_df = features_df.merge(erp_df, on=["Participant", "Index"], how="inner")
print(f"✅ Combined dataset (behavioral + ERP): {combined_df.shape}")

# === Step 4: Load onset features ===
onset_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/onset_vectors/onset_vectors"
X_onset, sentence_ids = load_onset_features(onset_dir)

df_onset = pd.DataFrame(X_onset)
df_onset["Sentence_ID"] = sentence_ids

# ✅ Fix: Remove 'SR' prefix from onset features for matching
df_onset["Sentence_ID"] = df_onset["Sentence_ID"].str.replace("SR", "", regex=False)

# Create Sentence_ID in combined_df
combined_df["Sentence_ID"] = combined_df["Index"].astype(str) + "_" + combined_df["Condition"]

# Merge onset features into dataset
df_with_onset = combined_df.merge(df_onset, on="Sentence_ID", how="inner").dropna()
print(f"✅ Final dataset (ERP + Onset): {df_with_onset.shape}")
display(df_with_onset.head(5))

# === Step 5: Prepare X and y for ML ===
# Keep only numeric columns for features
numeric_cols = df_with_onset.select_dtypes(include=[np.number]).drop(columns=["Index"]).columns
X = df_with_onset[numeric_cols].drop(columns=["Correct_Sequence (0/1)"]).values
y = df_with_onset["Correct_Sequence (0/1)"].values

print("✅ Final feature matrix:", X.shape)
print("✅ Target vector:", y.shape)


## Classification

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# === Step 1: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# === Step 2: Scale features ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === Step 3: Train Logistic Regression ===
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# === Step 4: Predict ===
y_pred_log = log_reg.predict(X_test_scaled)

# === Step 5: Evaluate ===
print("📊 Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log))

# Confusion Matrix
cm_log = confusion_matrix(y_test, y_pred_log)
sns.heatmap(cm_log, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# === Step 6: Feature Coefficients ===
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": log_reg.coef_[0]
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("🧠 Top 20 Most Influential Features (Logistic Regression):")
display(coef_df.head(20))



| Feature                   | Coefficient | Interpretation                                                |
| ------------------------- | ----------- | ------------------------------------------------------------- |
| **SpeechRate\_Preceding** | `+0.1138`   | ⬆️ Faster preceding speech increases likelihood of perception |
| **InterWordInterval**     | `–0.0406`   | ⬇️ Longer gap between words slightly reduces perception       |
| **N1\_amplitude**         | `–4.19e-6`  | ⬇️ More negative N1 is weakly associated with not perceiving  |
| **N280\_amplitude**       | `+1.83e-8`  | ⬆️ Positive N280 might weakly support perception              |


### XGBoost Classifier

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# === Step 1: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# === Step 2: Train XGBoost ===
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

# === Step 3: Predict ===
y_pred_xgb = xgb_model.predict(X_test)

# === Step 4: Evaluate ===
print("📊 XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

# Confusion Matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Greens", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.title("Confusion Matrix - XGBoost")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# === Step 5: Feature Importance ===
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("🔥 Top 20 Most Important Features (XGBoost):")
display(feature_importance.head(20))




## Inference ERP + Additional Features

In [ ]:
import re

# Load acoustic summary
acoustic_df = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Extracted_Features/summary_acoustic_features.csv")

# Remove extension and extract numeric part (e.g., from 'SR105.wav' get '105')
acoustic_df["file_id"] = acoustic_df["file"].str.replace(".wav", "", regex=False)
acoustic_df["Index"] = acoustic_df["file_id"].apply(lambda x: int(re.search(r"\d+", x).group()))

# Show result
acoustic_df.head()



In [ ]:
# ✅ Step 0: Ensure 'Index' is of the same type
combined_df["Index"] = combined_df["Index"].astype(int)
acoustic_df["Index"] = acoustic_df["Index"].astype(int)

# ✅ Step 1: Remove duplicate/conflicting columns
# If "file" and "file_id" are not needed, drop them to avoid merge conflicts
acoustic_df_clean = acoustic_df.drop(columns=["file", "file_id"], errors="ignore")

# Optional: Also remove any duplicate column names that already exist in combined_df
duplicate_cols = [col for col in acoustic_df_clean.columns if col in combined_df.columns and col != "Index"]
acoustic_df_clean = acoustic_df_clean.drop(columns=duplicate_cols)

# ✅ Step 2: Merge the feature sets
df_all = combined_df.merge(acoustic_df_clean, on="Index", how="left")

# ✅ Step 3: Clean up target column if needed
df_all["Correct_Sequence (0/1)"] = df_all["Correct_Sequence (0/1)"].astype(int)

# ✅ Step 4: Review the final dataset
print("✅ Final shape with all features:", df_all.shape)
display(df_all.head())



### Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 🎯 Define target variable
target = "Correct_Sequence (0/1)"

# 🧠 Define feature columns (exclude Participant, Index, and target)
exclude_cols = ["Participant", "Index", target]
feature_cols = [col for col in df_all.columns if col not in exclude_cols]

# 🧹 Drop any rows with missing values
df_model = df_all[feature_cols + [target]].dropna()

# 🪄 Split into train/test
X = df_model[feature_cols]
y = df_model[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🤖 Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 📊 Evaluate
y_pred = model.predict(X_test)
print("📋 Classification Report:")
print(classification_report(y_test, y_pred))

# 📉 Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


### XGBoost Classifier

In [ ]:
# ✅ Set the final merged dataset
df_final = df_all  # Your full dataset with acoustic + ERP + MFCC features

# === Step 1: Define features and target ===
X = df_final.drop(columns=["Correct_Sequence (0/1)", "Participant", "Index"], errors="ignore")
y = df_final["Correct_Sequence (0/1)"]

# === Step 2: Train-test split ===
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# === Step 3: Scale features ===
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === Step 4: Train XGBoost classifier ===
from xgboost import XGBClassifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# === Step 5: Evaluate model ===
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
y_pred = xgb_model.predict(X_test_scaled)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# === Confusion Matrix ===
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Perceived", "Perceived"])
disp.plot(cmap="Blues")

# === Step 6: Feature Importance ===
import pandas as pd
import matplotlib.pyplot as plt

importance_scores = xgb_model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_scores
}).sort_values(by="Importance", ascending=False)

print("\n🔝 Top 10 Most Important Features:")
print(importance_df.head(10))

# Plot top 15
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"][:15][::-1], importance_df["Importance"][:15][::-1])
plt.xlabel("Importance Score")
plt.title("Top 15 Most Important Features (XGBoost)")
plt.tight_layout()
plt.show()



Yes — acoustic features and ERP signals together can reliably predict perceptual outcomes.

The model successfully distinguishes perceived from not perceived items with 86% accuracy.

This supports the hypothesis that both neural (ERP) and speech timing/acoustic cues carry meaningful information about perception.

### SHAP

In [ ]:

# === Step 7: SHAP Interpretability ===
import shap

# Create SHAP explainer using the scaled training data
explainer = shap.Explainer(xgb_model, X_train_scaled)

# Compute SHAP values for the test set
shap_values = explainer(X_test_scaled)

# === Summary plot: Global feature impact ===
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns)


We applied SHAP analysis to interpret an XGBoost classifier trained to predict speech sequence perception. SHAP values indicated that both acoustic (ZCR, RMS), linguistic (FuncToWord2Interval), and ERP (N1, N280) features contributed substantially to prediction accuracy. Notably, longer function-to-word intervals and specific mid-range ERP amplitudes were associated with increased perception probability, while high-frequency signal irregularities (ZCR) decreased perception likelihood. These results validate the combined importance of low-level acoustic features and high-level neural responses in speech perception modeling.



### CROSS VALIDATION

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Features and target
X = df_all.drop(columns=["Correct_Sequence (0/1)", "Participant", "Index"], errors="ignore")
y = df_all["Correct_Sequence (0/1)"]

# Prepare cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n📂 Fold {fold} Classification Report:")
    print(classification_report(y_test, y_pred))

      # === Confusion Matrix with Cleaner Labels ===
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=["No", "Yes"],
                yticklabels=["No", "Yes"])
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    scores.append(model.score(X_test, y_test))
    fold += 1

print("\n✅ Average Accuracy Across Folds:", np.mean(scores))


Conlusion:

Balanced precision and recall across both perceived and non-perceived trials suggest the model does not favor one class over the other. The model consistently performs well across folds, indicating stable generalization. This shows that acoustic, prosodic, and ERP features together can reliably predict perception of the function word.

